# Milestone 2 — Bracket on Cora partitions

Compute $(\mathrm{lower}, \varepsilon, \mathrm{upper}, \mathrm{slack})$ for the label and 1-WL$(L=2)$ partitions, and visualise the bracket envelope.

## Setup

Resolve `onboarding/projects/` on `sys.path`, attach the reflection log, and import the canonical references (used only for spot-checks; the student version derives its own implementations).

In [ ]:
import os, sys
from pathlib import Path
_here = Path(os.getcwd()).resolve()
for _p in [_here, *_here.parents]:
    if (_p / 'shared' / '__init__.py').exists() and _p.name == 'projects':
        _PROJECTS = _p
        break
else:
    raise RuntimeError('could not locate onboarding/projects/')
_REPO = _PROJECTS.parent.parent
if str(_REPO) not in sys.path:
    sys.path.insert(0, str(_REPO))

import numpy as np
import matplotlib
matplotlib.use('Agg')   # headless for CI
import matplotlib.pyplot as plt

from onboarding.projects import reflect
reflect.start(notebook='m2', store=_PROJECTS / '.reflection.jsonl')
print(f'[setup] projects root: {_PROJECTS}')


In [ ]:
from onboarding.projects.shared import (
    Partition, label_partition, wl_partition,
    hbin, hbin_inverse, upper, lower, slack, verify, bracket_of,
    sum_partition, mean_partition, max_partition,
)
print('shared modules imported')


In [ ]:
from torch_geometric.datasets import Planetoid
_CACHE = Path.home() / '.cache' / 'planetoid'
_CACHE.mkdir(parents=True, exist_ok=True)
_cora = Planetoid(root=str(_CACHE / 'Cora'), name='Cora')[0]
n = int(_cora.num_nodes)
m = int(_cora.num_edges)
K = int(_cora.y.max().item()) + 1
print(f'Cora: n={n}, m={m}, classes={K}, x.shape={tuple(_cora.x.shape)}')


For multi-class Cora we **binarise per cell** ($e_C \leftarrow \min(e_C, 1-e_C)$) so the binary-entropy bracket applies.

In [ ]:
P_lbl = label_partition(_cora)
P_w2  = wl_partition(_cora, depth=2)
def stats(P):
    bin_e = np.array([min(e, 1-e) for e in P.e])
    q = np.array(P.q)
    eps = float(np.sum(q * bin_e))
    H   = float(np.sum(q * np.array([hbin(e) for e in bin_e])))
    return eps, H
for name, P in [('label', P_lbl), ('wl(L=2)', P_w2)]:
    eps, H = stats(P)
    lo, up = lower(H), upper(H)
    print(f'  {name:>8}: H={H:.4f}  ε={eps:.4f}  [lo={lo:.4f}, up={up:.4f}]  in_bracket={lo-1e-9 <= eps <= up+1e-9}')


**Gate M2.** Both partitions sit inside the bracket (lower ≤ ε ≤ upper).

In [ ]:
for name, P in [('label', P_lbl), ('wl(L=2)', P_w2)]:
    eps, H = stats(P)
    assert lower(H) - 1e-9 <= eps <= upper(H) + 1e-9, f'M2: bracket violated for {name}: ε={eps}, lower={lower(H)}, upper={upper(H)}'
print('[GATE OK] M2: both binarised Cora partitions lie inside the binary-entropy bracket')


In [ ]:
hs = np.linspace(0, 1, 401)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.fill_between(hs, [lower(h) for h in hs], [upper(h) for h in hs], color='C2', alpha=0.18, label='bracket envelope')
for name, P, mk in [('label', P_lbl, 'o'), ('wl(L=2)', P_w2, 's')]:
    eps, H = stats(P)
    ax.plot(H, eps, mk, ms=12, label=f'{name}: (H={H:.3f}, ε={eps:.3f})')
ax.set_xlabel('H(Y|Π)'); ax.set_ylabel('ε(Π)'); ax.legend(); ax.set_title('M2 — bracket envelope and Cora partitions')
_plots = _PROJECTS / 'capstone' / 'milestone2' / 'plots'; _plots.mkdir(exist_ok=True)
plt.tight_layout(); fig.savefig(_plots / 'm2_bracket.png', dpi=120); plt.show()


In [ ]:
reflect.log('M2.bracket', 'Cora label and 1-WL(L=2) partitions both in-bracket; plotted', 'HIGH')


**End of M2.**